### Deep Convolutional GAN(DCGAN):
- DCGAN are introduced to reduce the problem of mode collapse.
- Mode collapse occurs when the generator got biased towards a few outputs and can't able to produce outputs of every variation from the dataset.
- For example -
  - Take the mnist digit dataset(digits from 0 to 9), we want the generator should generate all type of digits but sometimes our generator got biased towards two to three digits and produce them only.
  - Because of that the discriminator also got optimized towards that particular digits only, and this state is known as mode collapse.
  - But this problem can be overcome by using Deep Convolutional (DCGAN).
- Here role of the discriminator is to determine that the image comes from either a real dataset or a generator.

### Architecture:
- The generator of the DCGAN architecture takes 100 uniform generated values using normal distribution as an input.
- First, it changes the dimension to 4x4x1024 and performed a fractionally stridden convolution 4 times with a stride of 1/2 [This means every time when applied, it doubles the image dimension while reducing the number of output channels].
- There are some architectural changes proposed in the generator such as the removal of all fully connected layers, and the use of Batch Normalization which helps in stabilizing training.
- In this paper, the authors use ReLU activation function in all layers of the generator, except for the output layers.
- The discriminator can be simply designed similar to a convolution neural network that performs an image classification task.
- However, the authors of this paper suggested some changes in the discriminator architecture.
- Instead of fully connected layers, they used only strided-convolutions with LeakyReLU as an activation function.
- The input of the generator is a single image from the dataset or generated image and the output is a score that determines whether the image is real or generated.

### Implementation:

In [1]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tqdm import tqdm
from IPython import display

# Check tensorflow version
print('Tensorflow version:', tf.__version__)

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
x_train = x_train.astype(np.float32) / 255.0
x_test = x_test.astype(np.float32) / 255.0
x_train.shape, x_test.shape

# We plot first 25 images of training dataset
plt.figgure(figsize=(10, 10))
for i in range(25):
    plt.subplot(5, 5, i+1)
    plt.xticks([])
    plt.yticks([])
    plt.imshow(x_train[i], cmap=plt.cm.binary)
plt.show()

batch_size = 32
# replacing the selected elements with new elements.
def create_batch(x_train):
    # Correct indentation here
    dataset = tf.data.Dataset.from_tensor_slices(x_train).shuffle(1000)
    # Combines consecutive elements of its dataset into batches
    dataset = dataset.batch(batch_size, drop_remainder=True).prefetch(1)
    # Creates a Dataset that prefetches elements from this dataset
    return dataset

# code
num_features = 100

generator = keras.models.Sequential([
    keras.layers.Dense(7 * 7 * 128, input_shape =[num_features]),
    keras.layers.Reshape([7, 7, 128]),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2DTranspose(
        64, (5, 5), (2, 2), padding =&quot;same&quot;, activation =&quot;selu&quot;),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2DTranspose(
        1, (5, 5), (2, 2), padding =&quot;same&quot;, activation =&quot;tanh&quot;),
])
generator.summary()

discriminator = keras.models.Sequential([
    keras.layers.Conv2D(64, (5,5), (2,2), padding=&quot;same&quot;, input_shape=[28, 28, 1]),
    keras.layers.LeakyReLU(0.2),
    keras.layers.Dropout(0.3),
    keras.layers.Conv2D(128, (5, 5), (2, 2), padding=&quot;same&quot;)
    keras.layers.LeakyReLU(0.2),
    keras.layers.Dropout(0.3),
    keras.layers.Flatten(),
    keras.layers.Dense(1, activation='sigmoid')
])
discrimator.summary()

# compile discriminator using binary cross entropy loss and adam optimizer
discriminator.compile(loss =&quot;binary_crossentropy&quot;, optimizer =&quot;adam&quot;)
# make  discriminator no-trainable as of  now
discriminator.trainable = False
# Combine both generator and discriminator
gan = keras.models.Sequential([generator, discriminator])
# compile generator using binary cross entropy loss and adam optimizer
gan.compile(loss =&quot;binary_crossentropy&quot;, optimizer =&quot;adam&quot;)


# Defining the GAN Model
seed = tf.random.normal(shape =[batch_size, 100])

def train_dcgan(gan, dataset, batch_size, num_features, epochs = 5):
    generator, discriminator = gan.layers
    for epoch in tqdm(range(epochs)):
        print()
        print(&quot;Epoch {}/{}&quot;.format(epoch + 1, epochs))

        for X_batch in dataset:
            # create a random noise of sizebatch_size * 100
            # to passit into the generator
            noise = tf.random.normal(shape =[batch_size, num_features])
            generated_images = generator(noise)

            # take batch of generated image and real image and
            #  use them to train  the discriminator
            X_fake_and_real = tf.concat([generated_images, X_batch], axis = 0)
            y1 = tf.constant([[0.]] * batch_size + [[1.]] * batch_size)
            discriminator.trainable = True
            discriminator.train_on_batch(X_fake_and_real, y1)

            # Here we will be training our GAN model, in this step
            #  we pass noise that uses generatortogenerate the image
            #  and pass it with labels as [1] So, it can fool the discriminator
            noise = tf.random.normal(shape =[batch_size, num_features])
            y2 = tf.constant([[1.]] * batch_size)
            discriminator.trainable = False
            gan.train_on_batch(noise, y2)

            # generate images for the GIF as we go
            generate_and_save_images(generator, epoch + 1, seed)

    generate_and_save_images(generator, epochs, seed)


def generate_and_save_images(model, epoch, test_input):
    # Indent this line properly to be part of the function
    predictions = model(test_input, training=False)

    fig = plt.figure(figsize=(10, 10))

    for i in range(25):
        plt.subplot(5, 5, i + 1)
        plt.imshow(predictions[i, :, :, 0] * 127.5 + 127.5, cmap='binary')
        plt.axis('off')

    # Save the generated images as a PNG file
    plt.savefig('image_epoch_{:04d}.png'.format(epoch))


# reshape to add a color map
x_train_dcgan = x_train.reshape(-1, 28, 28, 1) * 2. - 1.
# create batches
dataset = create_batch(x_train_dcgan)
# callthe training function with 10 epochs and record time %% time
train_dcgan(gan, dataset, batch_size, num_features, epochs = 10)


import imageio
import glob

anim_file = 'dcgan_results.gif'

with imageio.get_writer(anim_file, mode ='I') as writer:
  filenames = glob.glob('image*.png')
  filenames = sorted(filenames)
  last = -1
  for i, filename in enumerate(filenames):
    frame = 2*(i)
    if round(frame) &gt; round(last):
      last = frame
    else:
      continue
    image = imageio.imread(filename)
    writer.append_data(image)
  image = imageio.imread(filename)
  writer.append_data(image)
display.Image(filename = anim_file)

SyntaxError: invalid syntax (760386583.py, line 44)